# 1D-UNet + FiLM — Directly model trajectories (B, 2, T)

This notebook implements:
- UNet1D_FiLM: a 1D temporal UNet with FiLM conditioning (gamma, beta) at down/mid/up stages.
- Focus only on temporal relationships; no 2D image projection.
- A minimal runnable example: synthetic trajectories, a toy Flow Matching target, and a tiny training loop.

You can replace your previous 2D UNet pipeline (upsampler -> UNet2D -> downsampler) with this UNet1D_FiLM.
If later you want affordance, just pass an `afford` tensor of shape (B, D_a) to condition the model.


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Define FiLM and 1D-UNet body

In [8]:
class FiLMCond(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden=128, residual_init=True):
        super().__init__()
        self.to_gamma = nn.Sequential(
            nn.Linear(cond_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, feat_dim)
        )
        self.to_beta  = nn.Sequential(
            nn.Linear(cond_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, feat_dim)
        )
        if residual_init:
            nn.init.zeros_(self.to_gamma[-1].weight); nn.init.zeros_(self.to_gamma[-1].bias)
            nn.init.zeros_(self.to_beta[-1].weight);  nn.init.zeros_(self.to_beta[-1].bias)

    def forward(self, cond):
        gamma = self.to_gamma(cond)  # (B, C)
        beta  = self.to_beta(cond)   # (B, C)
        return gamma, beta


def apply_film(x, gamma, beta):
    # x: (B, C, T); gamma, beta: (B, C)
    return x * (1.0 + gamma.unsqueeze(-1)) + beta.unsqueeze(-1)


class FourierTimeEmbedding(nn.Module):
    """
    Map scalar time t in [0,1] (or noise level beta) to a vector of size time_dim.
    """
    def __init__(self, time_dim=64):
        super().__init__()
        assert time_dim % 2 == 0, "time_dim should be even"
        self.time_dim = time_dim
        half = time_dim // 2
        self.freq = nn.Parameter(torch.randn(1, half))
        self.proj = nn.Sequential(
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

    def forward(self, t):
        t = t.view(-1, 1)
        angles = 2 * math.pi * t @ self.freq  # (B, half)
        sin, cos = torch.sin(angles), torch.cos(angles)
        h = torch.cat([sin, cos], dim=-1)     # (B, time_dim)
        return self.proj(h)


class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=None):
        super().__init__()
        if p is None: p = (k - 1) // 2
        self.conv = nn.Conv1d(in_ch, out_ch, k, stride=s, padding=p)
        self.norm = nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch)
        self.act  = nn.SiLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x


class UNet1D_FiLM(nn.Module):
    def __init__(
        self,
        in_ch=2,          # e.g., (x,y)
        base=64,
        time_dim=64,
        afford_dim=0,     # >0 if you have affordance features
        depth=2,          # number of down/up stages
    ):
        super().__init__()
        self.in_ch = in_ch
        self.base = base
        self.time_emb = FourierTimeEmbedding(time_dim=time_dim)
        self.cond_dim = time_dim + afford_dim

        # input stem
        self.enc_in = ConvBlock1D(in_ch, base)

        # down path
        self.down_blocks = nn.ModuleList()
        self.down_film   = nn.ModuleList()
        ch = base
        for _ in range(depth):
            self.down_blocks.append(nn.Sequential(
                ConvBlock1D(ch, ch),
                ConvBlock1D(ch, ch*2),
            ))
            self.down_film.append(FiLMCond(self.cond_dim, ch*2))
            ch *= 2

        # mid
        self.mid_block = nn.Sequential(
            ConvBlock1D(ch, ch),
            ConvBlock1D(ch, ch),
        )
        self.mid_film = FiLMCond(self.cond_dim, ch)

        # up path
        self.up_blocks = nn.ModuleList()
        self.up_film   = nn.ModuleList()
        for _ in range(depth):
            # after concat with skip: channels become ch + ch//2
            self.up_blocks.append(nn.Sequential(
                ConvBlock1D(ch + ch//2, ch),
                ConvBlock1D(ch, ch//2),
            ))
            self.up_film.append(FiLMCond(self.cond_dim, ch//2))
            ch //= 2

        # output head
        self.out = nn.Conv1d(base, in_ch, kernel_size=1)

        # pool & up
        self.pool = nn.AvgPool1d(kernel_size=2, stride=2)
        self.up   = nn.Upsample(scale_factor=2, mode="linear", align_corners=False)

    def build_cond(self, t, afford=None):
        t_embed = self.time_emb(t)  # (B, time_dim)
        if afford is None:
            return t_embed
        return torch.cat([t_embed, afford], dim=-1)

    def forward(self, x, t, afford=None):
        """
        x: (B, in_ch, T)
        t: (B, 1)
        afford: (B, afford_dim) or None
        """
        cond = self.build_cond(t, afford)  # (B, cond_dim)

        # encoder
        skips = []
        h = self.enc_in(x)                 # (B, base, T)
        for block, film in zip(self.down_blocks, self.down_film):
            h = block(h)                   # (B, ch*2, T)
            gamma, beta = film(cond)
            h = apply_film(h, gamma, beta)
            skips.append(h)
            h = self.pool(h)

        # mid
        h = self.mid_block(h)
        gamma, beta = self.mid_film(cond)
        h = apply_film(h, gamma, beta)

        # decoder
        for block, film in zip(self.up_blocks, self.up_film):
            h = self.up(h)
            skip = skips.pop()
            # align T if pooling caused odd length
            if h.size(-1) != skip.size(-1):
                diff = skip.size(-1) - h.size(-1)
                if diff > 0:
                    h = F.pad(h, (0, diff))
                else:
                    h = h[..., :skip.size(-1)]
            h = torch.cat([h, skip], dim=1)  # channel concat
            h = block(h)
            gamma, beta = film(cond)
            h = apply_film(h, gamma, beta)

        return self.out(h)

## Minimal synthetic dataset (2D trajectories)

- Half samples: circle
- Half samples: line
- Returns x0 of shape (B, 2, T)


In [9]:
class ToyTrajectoryDataset(Dataset):
    def __init__(self, num_samples=1024, T=128):
        super().__init__()
        self.T = T
        ts = torch.linspace(0, 1, T)
        data = []
        for i in range(num_samples):
            if i % 2 == 0:
                # circle
                R = 1.0
                theta = 2 * math.pi * ts + torch.randn(1).item() * 0.2
                x = R * torch.cos(theta)
                y = R * torch.sin(theta)
            else:
                # line
                x0, x1 = torch.randn(1).item()*0.1, torch.randn(1).item()*0.1 + 1.0
                y0, y1 = torch.randn(1).item()*0.1, torch.randn(1).item()*0.1 + 1.0
                x = x0 + (x1 - x0) * ts
                y = y0 + (y1 - y0) * ts
            traj = torch.stack([x, y], dim=0)  # (2, T)
            data.append(traj)
        self.data = torch.stack(data, dim=0)  # (N, 2, T)

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        return self.data[idx]   # (2, T)


# sanity check
ds = ToyTrajectoryDataset(num_samples=64, T=128)
dl = DataLoader(ds, batch_size=16, shuffle=True)
batch = next(iter(dl))
batch.shape

torch.Size([16, 2, 128])

## Flow Matching toy target

We use a simple bridge:
x_t = alpha(t) * x0 + sigma(t) * eps,  eps ~ N(0, I)
with alpha(t) = 1 - t, sigma(t) = t

Then a simple proxy for target velocity:
v*(x_t, t) = d/dt x_t = -x0 + eps

This is only for demonstration. Replace with your actual FM target if needed.


In [4]:
def fm_sample_xt_and_vstar(x0):
    """
    Given clean trajectories x0: (B, 2, T), sample t in [0,1],
    build x_t and v_star (demo version).
    Returns: x_t, t, v_star
    """
    B = x0.size(0)
    t = torch.rand(B, 1, device=x0.device)
    eps = torch.randn_like(x0)
    alpha = 1.0 - t.view(B,1,1)
    sigma = t.view(B,1,1)
    x_t = alpha * x0 + sigma * eps
    v_star = -x0 + eps
    return x_t, t, v_star

## Tiny training loop (few steps just to verify)

In [5]:
model = UNet1D_FiLM(
    in_ch=2, base=64, time_dim=64, afford_dim=0, depth=2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

def train_steps(num_steps=20):
    model.train()
    ds = ToyTrajectoryDataset(num_samples=256, T=128)
    dl = DataLoader(ds, batch_size=32, shuffle=True, drop_last=True)

    it = iter(dl)
    losses = []
    for step in range(num_steps):
        try:
            x0 = next(it).to(device)     # (B,2,T)
        except StopIteration:
            it = iter(dl)
            x0 = next(it).to(device)

        x_t, t, v_star = fm_sample_xt_and_vstar(x0)
        v_pred = model(x_t, t)           # (B,2,T)

        loss = F.mse_loss(v_pred, v_star)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if (step+1) % 5 == 0:
            print(f"[step {step+1:02d}] loss = {loss.item():.6f}")
    return losses

losses = train_steps(num_steps=25)
losses[-1] if losses else None

RuntimeError: Given groups=1, weight of size [256, 384, 3], expected input[32, 512, 64] to have 384 channels, but got 512 channels instead

## Inference / sampling placeholder

For Flow Matching / diffusion sampling, typically:
- Initialize x_T ~ N(0, I) (or your terminal distribution)
- Reverse in time with an ODE-like update: x_{t-dt} = x_t + dt * v_theta(x_t, t)
- Use your scheduler and time grid as needed

Below is a one-step demo only:


In [ ]:
@torch.no_grad()
def one_step_reverse(x_t, t, dt=0.05):
    v_pred = model(x_t, t)  # (B,2,T)
    x_prev = x_t + dt * v_pred
    t_prev = torch.clamp(t - dt, 0.0, 1.0)
    return x_prev, t_prev

# demo from noise
B, T = 4, 128
x_noisy = torch.randn(B, 2, T, device=device)
t_now = torch.ones(B, 1, device=device) * 1.0
x_prev, t_prev = one_step_reverse(x_noisy, t_now, dt=0.1)
x_prev.shape, float(t_prev.min()), float(t_prev.max())